# Machine learning clásico que decide de verdad

**Facsímil 1 · Los cimientos** — capítulo 11 (machine learning clásico) y capítulo 2 (determinista frente
a probabilístico).

En la era de los LLM gigantes es fácil olvidar que **la mayoría de los problemas reales de empresa se
resuelven con modelos clásicos**: rápidos, baratos, que caben en un portátil y —ventaja enorme— que se
pueden *abrir y entender*. En este cuaderno construyes uno de principio a fin (un clasificador que lee
mensajes de soporte y decide si son **incidencia** o **consulta**), lo comparas con otros modelos, lo
mides de cinco formas distintas, y aprendes a leer *cómo* se equivoca y *por qué*. Saber cuándo basta con
lo clásico, en vez de disparar un cañón a una mosca, es de las señales de criterio que persigue el facsímil.

### Qué vas a aprender
- El **pipeline clásico** de texto: vectorizar con TF-IDF y clasificar.
- A leer una **matriz de confusión** (y a dibujarla), y por qué importa más que el «porcentaje de acierto».
- **Precisión, recall, F1**, el **umbral** y la **curva precision-recall**.
- A **comparar varios modelos** (regresión logística, Naive Bayes, SVM lineal) con criterio.
- El efecto de los **bigramas**, a **abrir el modelo** (interpretabilidad) y a hacer **análisis de errores**.

### Cómo se usa
Las celdas vienen **sin ejecutar**: pulsa ▶ de arriba abajo y verás nacer cada número y cada gráfico.

### Cuánto cuesta
Unos 18 minutos. CPU; sin GPU ni claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
# Colab ya lo trae. En tu maquina:  pip install scikit-learn numpy matplotlib
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
np.random.seed(0)
print("Listo.")


## 1. El pipeline clásico, en tres piezas

Un clasificador de texto clásico es una cadena de montaje: **vectorizar** (texto → números, con TF-IDF),
**clasificar** (un modelo que da una probabilidad por clase) y **decidir** (convertir esa probabilidad en
acción con un umbral). **TF-IDF** cuenta cuántas veces aparece cada palabra, pero **rebaja** las que salen
en todos los mensajes («de», «la») y **realza** las distintivas («error», «descuento»): el peso recae en
lo informativo.


In [ ]:
incidencias = ['La aplicación se cae cada vez que intento entrar, no puedo trabajar', 'El pago me da error y ya lo he intentado tres veces', 'No me llega el correo de confirmación y el pedido está parado', 'La web carga en blanco desde esta mañana, urgente', 'Me ha cobrado dos veces la misma factura, necesito que lo arregléis', 'El botón de descargar no hace nada, está roto', 'Llevo media hora y la página no responde, esto no va', 'Error 500 al guardar, pierdo todo el formulario', 'No puedo iniciar sesión, dice que mi contraseña es incorrecta y sí es correcta', 'La factura sale con el importe equivocado, está mal calculado', 'Se ha borrado mi pedido sin que yo hiciera nada', 'La sincronización falla y los datos no se actualizan', 'Me aparece la pantalla congelada y no avanza', 'El sistema rechaza mi tarjeta válida una y otra vez', 'Desde la última actualización la app va lentísima y se bloquea', 'No carga ninguna imagen del catálogo, todo roto', 'El enlace de recuperar contraseña está caído', 'Mi cuenta aparece suspendida sin motivo y no puedo acceder', 'La exportación a PDF da error y la necesito para hoy', 'Se cierra solo el programa cada dos por tres', 'El carrito pierde los productos al pasar a pagar', 'No funciona el buscador, devuelve siempre vacío', 'Quería comentar que llevo dos días sin recibir el pedido que ya pagué', 'Buenas, creo que la última factura está mal, me habéis cobrado de más', 'Hola, no sé si es normal, pero la aplicación se cierra sola a veces', 'Me gustaría saber por qué mi cuenta aparece bloqueada desde ayer', 'Perdón que insista, sigo sin poder acceder a mi pedido', 'Una duda, ¿por qué me ha llegado el artículo equivocado?']
consultas = ['Hola, ¿cómo puedo cambiar mi dirección de envío?', '¿Tenéis descuentos para estudiantes?', 'Quería saber si hacéis envíos a Canarias', '¿Cuánto tarda en llegar un pedido normalmente?', 'Me gustaría saber cómo darme de baja de la newsletter', '¿Es posible pagar a plazos?', 'Buenas, ¿dónde veo el histórico de mis facturas?', '¿Puedo cambiar el color después de comprar?', '¿Qué métodos de pago aceptáis?', 'Quería preguntar si el producto viene con garantía', '¿Cómo añado un segundo usuario a mi cuenta?', '¿Tenéis tienda física o solo online?', 'Me preguntaba si se puede regalar una suscripción', '¿Cada cuánto actualizáis el catálogo?', '¿Puedo recuperar una factura de hace dos años?', '¿Hay opción de factura para empresa con CIF?', 'Quería saber el horario de atención al cliente', '¿Se puede programar el envío para una fecha concreta?', '¿Ofrecéis prueba gratuita antes de pagar?', '¿Cómo cambio el idioma de la interfaz?', 'Buenas, ¿el precio incluye el IVA?', '¿Puedo tener varias direcciones guardadas?', '¿Puedo cambiar la fecha de entrega una vez hecho el pedido?', '¿Cómo funciona la devolución si el producto no me convence?', 'Quería saber si tenéis aplicación para el móvil', '¿El envío es gratis a partir de cierta cantidad?', '¿Puedo añadir una nota de regalo al pedido?', '¿Tenéis atención al cliente los fines de semana?']
textos = incidencias + consultas
y = np.array([1]*len(incidencias) + [0]*len(consultas))
print(f"{len(incidencias)} incidencias + {len(consultas)} consultas = {len(textos)} mensajes")
print("Algunos son AMBIGUOS a proposito (incidencias con tono de consulta), para que no sea trivial.")


## 2. De texto a números: TF-IDF

Ajustamos el vectorizador **solo con el train** (el vocabulario y los pesos IDF se aprenden de los datos
de entrenamiento, nunca del test; si no, habría fuga de información). Luego miramos qué palabras considera
más distintivas.


In [ ]:
X_tr_txt, X_te_txt, y_tr, y_te = train_test_split(textos, y, test_size=0.3, random_state=1, stratify=y)
vec = TfidfVectorizer()
X_tr = vec.fit_transform(X_tr_txt); X_te = vec.transform(X_te_txt)
nombres = np.array(vec.get_feature_names_out()); idf = vec.idf_
print("Vocabulario:", len(vec.vocabulary_), "palabras")
print("Mas distintivas (IDF alto):", ", ".join(nombres[np.argsort(idf)[-8:]]))
print("Mas comunes (IDF bajo):    ", ", ".join(nombres[np.argsort(idf)[:8]]))


## 3. Entrenar y leer la matriz de confusión

Entrenamos la **regresión logística**. Para evaluar no basta el acierto: usamos la **matriz de confusión**,
que cruza lo real con lo predicho. La casilla cara es **incidencia tratada como consulta** (un falso
negativo): un sistema roto que nadie atiende.


In [ ]:
clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
pred = clf.predict(X_te)
cm = confusion_matrix(y_te, pred, labels=[1,0])
print("                 pred:INCID   pred:CONSULTA")
print(f"  real INCIDENCIA     {cm[0,0]:>3}          {cm[0,1]:>3}   <- la casilla cara")
print(f"  real CONSULTA       {cm[1,0]:>3}          {cm[1,1]:>3}\n")
print(classification_report(y_te, pred, target_names=["consulta","incidencia"]))


## 4. La matriz de confusión, dibujada

Un mapa de calor lo hace más legible: la diagonal son los aciertos; fuera de ella, los dos tipos de error.


In [ ]:
fig,ax = plt.subplots(figsize=(3.8,3.4))
ax.imshow(cm, cmap="Greys")
ax.set_xticks([0,1]); ax.set_xticklabels(["incidencia","consulta"])
ax.set_yticks([0,1]); ax.set_yticklabels(["incidencia","consulta"])
ax.set_xlabel("predicho"); ax.set_ylabel("real")
for i in range(2):
    for j in range(2):
        ax.text(j,i,cm[i,j],ha="center",va="center",
                color="white" if cm[i,j]>cm.max()/2 else "black",fontsize=14)
ax.set_title("Matriz de confusion"); plt.tight_layout(); plt.show()


## 5. Precisión, recall y F1

- **Precisión** = de lo que marqué como incidencia, ¿qué fracción lo era? (fiabilidad de la alarma):
  $VP/(VP+FP)$.
- **Recall** = de las incidencias reales, ¿cuántas cacé? (cuántas se escapan): $VP/(VP+FN)$.
- **F1** = media armónica de ambas: $2PR/(P+R)$.

Casi siempre hay tensión: cazar todas (recall alto) implica más falsas alarmas (precisión baja). El
**umbral** regula ese equilibrio.


## 6. Mover el umbral según el coste del error

0.5 no tiene nada de especial. Si perder una incidencia es peor que molestar a una consulta, **bajamos el
listón**. Recorremos umbrales y miramos los dos errores.


In [ ]:
proba = clf.predict_proba(X_te)[:,1]
print("umbral | incidencias que se escapan (caro) | consultas molestadas (barato)")
for u in [0.5,0.35,0.2]:
    cm2 = confusion_matrix(y_te,(proba>=u).astype(int),labels=[1,0])
    print(f"  {u:>4}  |              {cm2[0,1]:>2}                   |          {cm2[1,0]:>2}")


## 7. La curva precision-recall

Cada umbral da un par (precisión, recall). Dibujarlos todos traza una curva: ves de un vistazo el
compromiso y eliges el punto que encaja con tu negocio.


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
pr, rc, _ = precision_recall_curve(y_te, proba)
ap = average_precision_score(y_te, proba)
plt.figure(figsize=(5.2,3.6)); plt.plot(rc, pr, color="black")
plt.xlabel("recall (incidencias cazadas)"); plt.ylabel("precision (alarmas correctas)")
plt.title(f"Curva precision-recall (AP={ap:.2f})"); plt.ylim(0,1.05); plt.tight_layout(); plt.show()
print("Cada punto es un umbral. Subir recall (cazar mas) suele bajar precision (mas falsas alarmas).")


## 8. Abrir el modelo: ¿qué palabras delatan a una incidencia?

Un clasificador clásico **se lee por dentro**: sus coeficientes dicen qué palabras empujan hacia cada
clase. Interpretabilidad de verdad, no aproximada.


In [ ]:
pesos = clf.coef_[0]
print("Mas INCIDENCIA:", ", ".join(nombres[np.argsort(pesos)[-10:]][::-1]))
print("Mas CONSULTA:  ", ", ".join(nombres[np.argsort(pesos)[:10]]))


## 9. ¿Es bueno el modelo elegido? Comparar varios

La regresión logística no es la única opción. Comparamos tres clásicos sobre los mismos datos, con
**validación cruzada** (más robusta que un único split): regresión logística, **Naive Bayes** (rápido,
clásico para texto) y **SVM lineal** (potente en alta dimensión).


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
modelos = {
    "Regresion logistica": LogisticRegression(max_iter=1000),
    "Naive Bayes":         MultinomialNB(),
    "SVM lineal":          LinearSVC(),
}
print(f"{'modelo':<22}{'acierto (validacion cruzada)':>30}")
for nombre, m in modelos.items():
    sc = cross_val_score(make_pipeline(TfidfVectorizer(), m), textos, y, cv=4)
    print(f"{nombre:<22}{sc.mean():>22.3f} ± {sc.std():.3f}")
print("\nLos tres son razonables; ninguno es 'el mejor' siempre. Comparar es parte del oficio.")


## 10. ¿Ayudan los bigramas?

Por defecto TF-IDF mira palabras sueltas (unigramas). Con **bigramas**, «no funciona» cuenta como una
unidad, lo que a veces captura mejor el sentido. Lo medimos.


In [ ]:
for ngram, etiqueta in [((1,1),"unigramas"),((1,2),"uni+bigramas")]:
    sc = cross_val_score(make_pipeline(TfidfVectorizer(ngram_range=ngram), LogisticRegression(max_iter=1000)), textos, y, cv=4)
    print(f"{etiqueta:<14}: acierto {sc.mean():.3f}")
print("Mas contexto (bigramas) a veces ayuda y a veces no; con corpus pequeno, poco margen.")


## 11. Análisis de errores: mirar lo que falla

El paso que más enseña y que casi nadie hace: leer los mensajes mal clasificados. Suelen revelar los casos
ambiguos (o que el modelo «acierta por el motivo equivocado»).


In [ ]:
mal = np.where(pred != y_te)[0]
print(f"{len(mal)} mensajes mal clasificados de {len(y_te)}:\n")
for i in mal:
    real = "incidencia" if y_te[i]==1 else "consulta"
    print(f"  [{proba[i]:.2f}] real={real:<10} -> {X_te_txt[i][:70]}")
print("\nMira si son casos genuinamente ambiguos (tono de consulta + problema real).")


## 12. La línea base y la curva de aprendizaje

Un número de acierto no significa nada sin algo con qué compararlo. La línea base: un modelo tonto que
siempre predice la clase mayoritaria. Y la **curva de aprendizaje** dice si etiquetar más datos rentaría.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import learning_curve
tonto = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
print(f"Modelo tonto:  {tonto.score(X_te,y_te):.3f}   |   nuestro modelo: {clf.score(X_te,y_te):.3f}")
tam, _, te_sc = learning_curve(make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000)),
                               textos, y, cv=4, train_sizes=np.linspace(0.3,1.0,6))
plt.figure(figsize=(6.3,3.2)); plt.plot(tam, te_sc.mean(1), "-o", color="black")
plt.xlabel("nº de mensajes de entrenamiento"); plt.ylabel("acierto en validacion")
plt.title("Curva de aprendizaje: ¿cuantos datos rentan?"); plt.tight_layout(); plt.show()


## 13. Pruébalo tú

1. **Tu propio mensaje:** `clf.predict_proba(vec.transform(["la app no carga, tengo una reunion ya"]))[0,1]`.
2. **Mensaje tramposo:** «quería saber por qué la app se ha caído» (tono de consulta + problema). ¿Hacia
   dónde se inclina?
3. **Pondera las clases** (`class_weight='balanced'`): ¿cómo cambia la matriz de confusión?
4. **Prueba un modelo más** (RandomForest): ¿gana a los lineales en este texto? (Pista: no siempre.)


## 14. Errores comunes

- **Confundir acierto con utilidad.** Con clases desbalanceadas, «siempre la mayoritaria» acierta mucho y
  es inútil. Por eso comparamos con el tonto.
- **Ajustar el vectorizador con todo el dataset** antes de partir: fuga de información, resultados falsos.
- **Quedarse con el umbral 0.5** cuando los errores no cuestan lo mismo.
- **No mirar los errores ni los coeficientes.** A menudo revelan que el modelo «acierta por el motivo
  equivocado».


## 15. Qué te llevas

- Un modelo **clásico** (TF-IDF + un clasificador lineal) resuelve de sobra muchísimos problemas de texto,
  es rápido, barato y **se entiende por dentro**.
- La **matriz de confusión**, la **curva precision-recall** y el **umbral** cuentan *cómo* fallas; el
  acierto solo, no.
- **Comparar modelos**, mirar **bigramas**, **analizar los errores** y tener una **línea base** son hábitos
  baratos que separan el trabajo serio de la demo bonita.

**Para seguir:** el cap. 9 cambia TF-IDF por *embeddings* (texto a vectores con significado); el facsímil 7
va de evaluar y calibrar estas decisiones; y el facsímil 4 te enseña cuándo subir de lo clásico a un LLM...
y cuándo no.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*